# Lesson 1 — first table · ตารางแรก

LanceDB ไม่มี server ตารางหนึ่งคือ directory หนึ่ง
เขียนหนึ่งครั้ง ได้ไฟล์สามอย่าง `.txn` · `.manifest` · data fragment
บทนี้สร้างตาราง เพิ่มแถว แล้วเปิด directory ดูว่าเกิดอะไรขึ้นจริง

In [1]:
%pip install -q lancedb pandas

Note: you may need to restart the kernel to use updated packages.


`connect` แค่ชี้ไปที่ folder ยังไม่เกิดไฟล์อะไรทั้งนั้น
ไม่มี socket ไม่มี port สอง process เปิด folder เดียวกันได้

In [2]:
import lancedb

db = lancedb.connect("./data/lesson1")

ส่ง list ของ dict เข้าไป schema เดาให้เอง
type ที่ได้เป็น Arrow type (`int64` `string`) ไม่ใช่ Python type

In [3]:
rows = [
    {"id": 1, "repo": "lance-indexer", "lang": "ts", "stars": 3},
    {"id": 2, "repo": "session-dream", "lang": "ts", "stars": 7},
    {"id": 3, "repo": "arra-memory-py", "lang": "py", "stars": 1},
]
tbl = db.create_table("repos", data=rows, mode="overwrite")
tbl.schema

id: int64
repo: string
lang: string
stars: int64

In [4]:
tbl.to_pandas()

,id,repo,lang,stars
0,1,lance-indexer,ts,3
1,2,session-dream,ts,7
2,3,arra-memory-py,py,1


`where` รับ string หน้าตาเหมือน SQL
เงื่อนไขถูกดันลงไปตอนอ่านไฟล์ ไม่ได้อ่านทั้งหมดขึ้นมาแล้วค่อยกรอง

In [5]:
tbl.search().where("lang = 'ts'").to_pandas()

,id,repo,lang,stars
0,1,lance-indexer,ts,3
1,2,session-dream,ts,7


`add` ไม่แก้ไฟล์เดิม เขียน fragment ใหม่ต่อท้าย แล้วออก manifest ใหม่
version เลยขยับจาก 1 เป็น 2

In [6]:
tbl.add([{"id": 4, "repo": "lanceglass", "lang": "ts", "stars": 2}])
print("count:", tbl.count_rows(), "| version:", tbl.version)

count: 4 | version: 8


เปิด directory ดู
`create` หนึ่งครั้ง `add` หนึ่งครั้ง ก็ควรเห็น txn 2 · manifest 2 · fragment 2

ถ้ารัน notebook นี้ซ้ำ จะเห็นมากกว่านั้น
เพราะ `mode="overwrite"` ไม่ได้ลบของเก่า แค่ออก manifest ใหม่ที่ไม่ชี้ไปหามัน
ไฟล์เก่ายังอยู่ จนกว่าจะสั่ง cleanup เอง (บทที่ 5)

In [7]:
!find data/lesson1 -type f | sort

/Users/beta/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/pty.py:95: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  pid, fd = os.forkpty()


Exception in thread 

LanceDBBackgroundEventLoop

:


Traceback (most recent call last):


...

^

OSError

: 

[Errno 9] Bad file descriptor

data/lesson1/repos.lance/_transactions/0-f107dbee-6b04-4703-9029-c25aa5331023.txn
data/lesson1/repos.lance/_transactions/1-613fb730-2071-48e0-9249-1be6d95068e2.txn
data/lesson1/repos.lance/_transactions/2-495d683e-ac26-47e1-beb6-fa6a646c8f8a.txn
data/lesson1/repos.lance/_transactions/3-93bac97a-718e-4c33-a75a-2197c538064e.txn
data/lesson1/repos.lance/_transactions/4-d1a414db-9203-4dac-aff7-13aa74e972a7.txn
data/lesson1/repos.lance/_transactions/5-14db9487-61ec-4a58-aa73-b5d5c0fc8736.txn
data/lesson1/repos.lance/_transactions/6-326d86f3-7cab-4242-9f0d-caafd708ddcc.txn
data/lesson1/repos.lance/_transactions/7-c6aab066-4cf0-404e-98e5-8894e9f71980.txn
data/lesson1/repos.lance/_versions/18446744073709551607.manifest
data/lesson1/repos.lance/_versions/18446744073709551608.manifest
data/lesson1/repos.lance/_versions/18446744073709551609.manifest
data/lesson1/repos.lance/_versions/18446744073709551610.manifest
data/lesson1/repos.lance/_versions/18446744073709551611.manifest
data/lesson1/repos.